# NSTX / NSTX-U — profile gradient scaling (`omt` / `omne`)

The NSTX counterpart of `DIIID_verify/diiid_scaling_check.ipynb`, same structure,
one method instead of two. On DIII-D 162940 the gradient path is the one that
behaved: it reproduces the source exactly at `alpha = 1.0` — an `mtanh_full`
scaling does not, because it replaces the profile with its own fit even at unity
— and it needs no fit to succeed before the scaling means anything.

| Method | Model | What the knob does |
|--------|-------|--------------------|
| `apply_omt` / `apply_omne` | power law on the log-gradient | re-exponentiates the profile about its value at `rhot_midped`, with the exponent ramped from 1 in the core to `alpha` outboard of `rhot_topped` |

The ramp and the pivot are the whole story, so they are per discharge and they
are the first thing to edit: `RAMP_WINDOWS` below. Every comparison is drawn
twice, full radius and `0.8 - 0.99`, because a pedestal change is a few percent
of a core-scaled axis and is only legible zoomed.

## Setup

In [ ]:
import glob
import os

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from TPED.projects.discharge_tools.src.transforms.pedestal_transforms import (
    apply_omt, apply_omne)
from TPED.projects.discharge_tools.src.discharge_data import DischargeData
from TPED.projects.discharge_tools.src.discharge_physics import DischargePhysics

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

### Case and knobs — edit these

`RAMP_WINDOWS[shot] = (rhot_topped, rhot_midped)` sets the exponent ramp

    alpha_profile = 1 + (alpha - 1) * (tanh((rhot - topped)/(midped - topped)) + 1)/2

and the power law is pivoted on the profile's value at `rhot_midped`:

- **`midped = 1.0`** pivots on the separatrix, so separatrix values are held and
  the profile fans inboard of it. This is what the handed-over IFS `modProfs`
  scan for DIII-D 162940 used (`rhotMidPed = 1.0`, `rhotTopPed = 0.8`), and it is
  the default here so the two campaigns are comparable.
- **`topped`** is where the ramp reaches half its travel. The transform is the
  identity well inboard of it — at `topped = 0.8` the exponent is 3e-4 of the way
  to `alpha` at the axis, so the core is untouched by construction.

A window measured as a *pedestal* is usually the wrong thing here: 162940's
mtanh pedestal is 0.017 wide, which makes `alpha_profile` exactly 1 everywhere
inboard of 0.94, so every alpha produced the same profile and the first campaign
could not move the equilibrium. Section 2 suggests a window from each
discharge's own gradient, so you can see what the data says before overriding.

In [ ]:
# First path that exists wins, so this runs on NERSC and on a laptop.
DISCHARGE_ROOT_CANDIDATES = [
    r"/global/homes/j/joeschm/data/ST_research/NSTXU_discharges",   # NERSC
    r"C:/Users/joesc/git/ST_research/NSTXU_discharges",             # local
]

SHOTS = [129015, 129038, 132543, 132588]

# 129038's directory holds five pfiles, so auto-discovery refuses to guess.
PFILES = {129038: "p129038.00400"}

# (rhot_topped, rhot_midped) per discharge. Start from the IFS handoff pair and
# override per shot once section 2 tells you the pedestal sits somewhere else.
DEFAULT_WINDOW = (0.8, 1.0)
RAMP_WINDOWS = {
    129015: (0.8, 1.0),
    129038: (0.8, 1.0),
    132543: (0.8, 1.0),
    132588: (0.8, 1.0),
}

# For the record: pe pedestals measured from an mtanh_full fit, where one exists.
# NOT used for scaling -- see the note above about narrow windows.
MEASURED_PE_WINDOWS = {
    129015: (0.885, 0.926),        # full pe width 0.081, fit rms 0.29%
}

ALPHAS = [0.7, 0.8, 0.9, 1.0, 1.1]      # matches the DIII-D campaign's points

# Hatch's 129015 runs scale Te and ne SEPARATELY: profiles_*_1.3T moves the
# temperatures only, profiles_*_1.3n the densities only. Section 5 measures what
# "1.3" actually means from his own files rather than assuming it; HATCH_CASES is
# only the guess its table is read against.
HATCH_ROOT_CANDIDATES = [
    os.environ.get("HATCH_ROOT", ""),                                # override
    "/pscratch/sd/j/joeschm/cheaseBS_hatch_results/for_joey",        # NERSC
]
HATCH_SHOT = 129015
HATCH_CASES = {"1.3T": (1.3, 1.0), "1.3n": (1.0, 1.3)}   # (alpha_T, alpha_n)

RHO_FULL = (0.0, 1.0)                   # full-radius view
RHO_PED = (0.8, 0.99)                   # where the pedestal change is legible

TZ_EQ_TI = True                         # IFS `set_tz_eq_ti`; apply_omt's apply_to_tz

RUN_CHEASEBS = False                    # section 6 only prints commands while False

In [ ]:
def discharge_root():
    for d in DISCHARGE_ROOT_CANDIDATES:
        if os.path.isdir(d):
            return d
    raise FileNotFoundError(
        "no discharge root found; add this machine's path to "
        f"DISCHARGE_ROOT_CANDIDATES (tried {DISCHARGE_ROOT_CANDIDATES})")


def load(shot):
    d = os.path.join(discharge_root(), str(shot))
    kw = {"input_dir": d}
    if shot in PFILES:
        kw["pfile"] = os.path.join(d, PFILES[shot])
    return DischargePhysics(DischargeData(**kw))


PHYS = {s: load(s) for s in SHOTS}

# Which of the scaled variables each discharge actually carries. 129015 and
# 129038 have no nz/Tz: apply_omne CREATES nz from quasineutrality there, so a
# density scaling on those shots writes a species the base profiles did not have.
print(f"{'shot':<8}{'nodes':>6}{'rhot range':>16}   variables present")
for s, p in PHYS.items():
    r = p.rhot.values
    have = [v for v in ("Te", "Ti", "Tz", "ne", "ni", "nz") if v in p.ds]
    print(f"{s:<8}{len(r):>6}{f'{r[0]:.3f} - {r[-1]:.3f}':>16}   {', '.join(have)}")

### Plotting helper

`compare_profiles` is the DIII-D notebook's, with an `axes` argument added so it
can draw into a grid. `compare_grid` stacks it: one row full radius, one row
`RHO_PED`, so every comparison is shown both ways without having to remember to
re-run it with a different range.

In [ ]:
def get_vals(phys, var):
    """Extract numpy array, stripping pint units if present."""
    da = phys.ds[var]
    return da.pint.magnitude if hasattr(da, "pint") else da.values


def compare_profiles(phys_list, labels, vars=("Te", "ne"), title="",
                     rho_range=(0.0, 1.0), axes=None):
    """Overlay profiles from multiple DischargePhysics objects."""
    own = axes is None
    if own:
        fig, axes = plt.subplots(1, len(vars), figsize=(4.5 * len(vars), 4))
    axes = np.atleast_1d(axes)
    colors = plt.cm.tab10(np.linspace(0, 0.8, len(phys_list)))
    linestyles = ["-"] + ["--", "-.", ":"] * len(phys_list)

    for p, label, color, ls in zip(phys_list, labels, colors, linestyles):
        rhot = p.rhot.values
        mask = (rhot >= rho_range[0]) & (rhot <= rho_range[1])
        for ax, var in zip(axes, vars):
            if var not in p.ds:
                continue
            ax.plot(rhot[mask], get_vals(p, var)[mask], label=label,
                    color=color, ls=ls, lw=1.4)
    for ax, var in zip(axes, vars):
        ax.set_xlabel("rho_tor")
        ax.set_title(var)
        ax.grid(alpha=0.3)
    axes[0].legend(fontsize=7)
    if own:
        plt.suptitle(title)
        plt.tight_layout()
    return axes


def compare_grid(phys_list, labels, vars, title="", ped=None):
    """The same overlay twice: full radius on top, the pedestal view below."""
    ped = ped or RHO_PED
    fig, axes = plt.subplots(2, len(vars), figsize=(4.0 * len(vars), 7.0),
                             squeeze=False)
    compare_profiles(phys_list, labels, vars=vars, rho_range=RHO_FULL,
                     axes=axes[0])
    compare_profiles(phys_list, labels, vars=vars, rho_range=ped, axes=axes[1])
    for ax in axes[1]:
        ax.set_title(f"{ax.get_title()}   [{ped[0]}-{ped[1]}]", fontsize=9)
    fig.suptitle(title)
    fig.tight_layout()
    return fig


def scaled(shot, omt=None, omn=None):
    """One scaled discharge, using that shot's own ramp window."""
    topped, midped = RAMP_WINDOWS.get(shot, DEFAULT_WINDOW)
    p = PHYS[shot]
    if omt is not None:
        p = p.apply_omt(alpha=omt, rhot_midped=midped, rhot_topped=topped,
                        apply_to_tz=TZ_EQ_TI)
    if omn is not None:
        p = p.apply_omne(alpha=omn, rhot_midped=midped, rhot_topped=topped)
    return p


def present(shot, vars):
    """Only the variables this discharge actually has, in the order given."""
    return [v for v in vars if v in PHYS[shot].ds]

### Window check — what the data says

Fit-free, deliberately crude, and only a suggestion: `midped_hint` is where
`|d ne/d rhot|` peaks in the outer half, and `topped_hint` is the first point
inboard of it where the gradient has fallen to a quarter of that peak. Compare it
against what `RAMP_WINDOWS` is set to — and remember that a window this narrow is
the failure mode described above. The hint says where the pedestal *is*, not what
the ramp *should be*.

In [ ]:
def suggest_window(phys, var="ne"):
    """(topped_hint, midped_hint) from the steepest gradient. Fit-free."""
    rhot = phys.rhot.values
    y = get_vals(phys, var)
    g = np.abs(np.gradient(y, rhot))
    outer = rhot > 0.5
    idx = int(np.arange(len(rhot))[outer][np.argmax(g[outer])])
    midped = float(rhot[idx])
    topped = max(midped - 0.05, 0.0)
    for j in range(idx, -1, -1):
        if g[j] < 0.25 * g[idx]:
            topped = float(rhot[j])
            break
    if not 0.0 <= topped < midped:
        topped = max(midped - 0.05, 0.0)
    return topped, midped


print(f"{'shot':<8}{'in use':>16}{'ne gradient hint':>20}{'measured pe':>16}")
for s in SHOTS:
    used = RAMP_WINDOWS.get(s, DEFAULT_WINDOW)
    hint = suggest_window(PHYS[s])
    meas = MEASURED_PE_WINDOWS.get(s)
    print(f"{s:<8}{f'{used[0]:.3f}/{used[1]:.3f}':>16}"
          f"{f'{hint[0]:.3f}/{hint[1]:.3f}':>20}"
          f"{(f'{meas[0]:.3f}/{meas[1]:.3f}' if meas else '-'):>16}")

### How big is the change, really

`L = -y/(dy/drho)` at the pedestal, and the thermal pressure integrated over
rho. A scaling that leaves `dp_int` at zero leaves the equilibrium alone, and
cheaseBS will hand back the profile it started from.

In [ ]:
EV = 1.602176634e-19


def pressure(p):
    out = get_vals(p, "ne") * get_vals(p, "Te")
    for n, t in (("ni", "Ti"), ("nz", "Tz")):
        if n in p.ds and t in p.ds:
            out = out + get_vals(p, n) * get_vals(p, t)
    return out * EV


def summary(base, cases, labels, radius=0.95):
    """One row per case: L at `radius`, and pressure change vs base."""
    x = base.rhot.values
    o = np.argsort(x)
    p0 = pressure(base)
    L0 = np.asarray(base.gradient_length("Te").values, float)
    p0_int = float(np.trapezoid(p0[o], x[o]))
    L0_r = float(np.interp(radius, x[o], L0[o]))

    print(f"{'case':<22}{'L_Te':>9}{'L/L_base':>10}{'dp@0.5':>9}{'dp_int':>9}")
    print(f"{'base':<22}{L0_r:>9.4f}{1.0:>10.3f}{'0.00%':>9}{'0.00%':>9}")
    for p, label in zip(cases, labels):
        L = np.asarray(p.gradient_length("Te").values, float)
        L_r = float(np.interp(radius, x[o], L[o]))
        pr = pressure(p)
        d_mid = np.interp(0.5, x[o], pr[o]) / np.interp(0.5, x[o], p0[o]) - 1.0
        d_int = float(np.trapezoid(pr[o], x[o])) / p0_int - 1.0
        print(f"{label:<22}{L_r:>9.4f}{L_r / L0_r:>10.3f}"
              f"{100 * d_mid:>8.2f}%{100 * d_int:>8.2f}%")

---
## 1. `apply_omt` — temperature gradient scaling

`alpha < 1` flattens, `alpha > 1` steepens, and `alpha = 1.0` is the null test:
it must return the source profile exactly. `Tz` follows `Ti` when the discharge
has one (`TZ_EQ_TI`).

In [ ]:
for s in SHOTS:
    cases = [scaled(s, omt=a) for a in ALPHAS]
    top, mid = RAMP_WINDOWS.get(s, DEFAULT_WINDOW)
    compare_grid(cases, [f"alpha={a}" for a in ALPHAS],
                 vars=present(s, ("Te", "Ti", "Tz")),
                 title=f"{s} — apply_omt   topped={top}, midped={mid}")
    plt.show()

## 2. `apply_omne` — density gradient scaling

`ni` is scaled by the same factor as `ne` and `nz` is rebuilt from
quasineutrality, so watch the printed error. On a discharge with no `nz` in the
base profiles this *creates* one.

In [ ]:
for s in SHOTS:
    cases = [scaled(s, omn=a) for a in ALPHAS]
    top, mid = RAMP_WINDOWS.get(s, DEFAULT_WINDOW)
    compare_grid(cases, [f"alpha={a}" for a in ALPHAS],
                 vars=present(s, ("ne", "ni", "nz")),
                 title=f"{s} — apply_omne   topped={top}, midped={mid}")
    plt.show()

    print(f"{s} quasineutrality error, base: {PHYS[s].check_quasineutrality():.2e}")
    for a, p in zip(ALPHAS, cases):
        print(f"   alpha={a}: {p.check_quasineutrality():.2e}")

## 3. Both knobs together

What the cheaseBS cases in section 5 use: `omt` and `omne` on the diagonal,
which is the axis the campaign is about.

In [ ]:
for s in SHOTS:
    cases = [scaled(s, omt=a, omn=a) for a in ALPHAS]
    top, mid = RAMP_WINDOWS.get(s, DEFAULT_WINDOW)
    compare_grid(cases, [f"alpha={a}" for a in ALPHAS],
                 vars=present(s, ("Te", "ne", "Ti", "ni")),
                 title=f"{s} — apply_omt + apply_omne   topped={top}, midped={mid}")
    plt.show()

    print(f"--- {s} ---")
    summary(PHYS[s], cases, [f"omt=omne={a}" for a in ALPHAS])

## 4. Requested `alpha` vs achieved gradient ratio

`alpha` is the *asymptotic* exponent, not the gradient ratio. Differentiating
`ln y_new = ln y_pivot + alpha_prof(rho) * ln(y/y_pivot)` gives

    omega_new/omega_base = alpha_prof - ln(y/y_pivot) * (d alpha_prof/d rho) / omega_base

The second term lives entirely inside the ramp and its sign is set by
`d alpha_prof/d rho`, so for `alpha < 1` the ramp region comes out **steeper**
even though the request was to flatten. On DIII-D 162940 at `alpha = 0.7`,
`topped = 0.8`, the achieved `ne` ratio peaks at 3.4 near `rho = 0.7` and only
drops below 1 outboard of about 0.93.

Read this table before quoting a scan point as "omt = 0.7". The directory tag is
the requested alpha and nothing else — `tag_of()` formats the number that was
asked for, and nothing in the pipeline measures what came out.

The ratio prints as `n/a` where the base gradient is too small to divide by. That
is not a rare guard on these discharges: NSTX core density profiles are flat or
hollow, so `omega_ne` crosses zero inboard of the pedestal and the ratio there is
meaningless rather than large.

In [ ]:
PROBE = [0.5, 0.7, 0.8, 0.85, 0.9, 0.95, 0.99]


def omega(p, var):
    """-d ln y / d rho, the normalised log-gradient the alphas are named for."""
    rhot = p.rhot.values
    return -np.gradient(np.log(get_vals(p, var)), rhot)


# Below this fraction of the profile's own peak gradient, omega_base is
# indistinguishable from zero and the ratio is noise rather than a large number.
OMEGA_FLOOR_FRAC = 0.05


def omega_ratio(shot, var, alpha):
    """omega_new/omega_base, NaN where omega_base is too small to divide by."""
    base = PHYS[shot]
    is_t = var.startswith("T")
    p = scaled(shot, omt=alpha if is_t else None, omn=None if is_t else alpha)
    ob, on = omega(base, var), omega(p, var)
    floor = OMEGA_FLOOR_FRAC * np.nanmax(np.abs(ob))
    return np.where(np.abs(ob) < floor, np.nan, on / ob)


def achieved_table(shot, alpha, vars=("Te", "ne")):
    base = PHYS[shot]
    rhot = base.rhot.values
    idx = [int(np.abs(rhot - x).argmin()) for x in PROBE]
    top, mid = RAMP_WINDOWS.get(shot, DEFAULT_WINDOW)
    w = (np.tanh((rhot - top) / (mid - top)) + 1) / 2
    nominal = 1 + (alpha - 1) * w

    print(f"--- {shot}   alpha={alpha}   topped={top}, midped={mid} ---")
    print("%-26s %s" % ("rho_tor", " ".join("%7.2f" % x for x in PROBE)))
    print("%-26s %s" % ("alpha_profile (nominal)",
                        " ".join("%7.3f" % nominal[k] for k in idx)))
    for var in vars:
        if var not in base.ds:
            continue
        r = omega_ratio(shot, var, alpha)
        cells = ["%7.3f" % r[k] if np.isfinite(r[k]) else "%7s" % "n/a"
                 for k in idx]
        print("%-26s %s" % (f"achieved ratio, {var}", " ".join(cells)))


for s in SHOTS:
    achieved_table(s, 0.7)

---
## 5. Mimic Hatch's individual T and n scalings

His 129015 runs move one family at a time -- `profiles_*_1.3T` is temperature
only, `profiles_*_1.3n` density only -- so this section measures his transform
before reproducing it.

For a power law about the separatrix value the exponent is recoverable directly
from the file pair:

    alpha_prof(rho) = ln(y_scaled / y_scaled_sep) / ln(y_base / y_base_sep)

and the ramp is fitted to that. The **flat value ratio** is printed beside the
fit as the competing hypothesis: if "1.3" means `y -> 1.3*y` rather than
`omega -> 1.3*omega`, that column reads 1.30 with near-zero spread while the fit
rms is poor. Read both before believing either. A species the set did not touch
comes back as `alpha = 1.0`, which is how the T-only / n-only split shows up.

The fitter was checked against a synthetic pair built with known parameters
(alpha 1.3, topped 0.8, midped 1.0) on this discharge: it returns those three to
four decimals and 1.0000 for the untouched species.

In [ ]:
from scipy.optimize import least_squares


def hatch_root():
    for d in HATCH_ROOT_CANDIDATES:
        if d and os.path.isdir(d):
            return d
    return None


def hatch_sets(d):
    """{suffix: {spec: path}} for one directory, or {} if it is not a run.

    A run holds the bare profiles_{e,i,z} plus at least one suffixed set; the
    bare set is the reference the scaled ones are measured against.
    """
    out = {}
    for path in sorted(glob.glob(os.path.join(d, "profiles_e*"))):
        suffix = os.path.basename(path)[len("profiles_e"):]
        trio = {sp: os.path.join(d, f"profiles_{sp}{suffix}")
                for sp in ("e", "i", "z")}
        if all(os.path.isfile(p) for p in trio.values()):
            out[suffix] = trio
    return out if "" in out and len(out) > 1 else {}


def hatch_runs(root):
    """{run name: sets} for root itself, else for each of its subdirectories."""
    own = hatch_sets(root)
    if own:
        return {os.path.basename(os.path.normpath(root)): own}
    found = {}
    for entry in sorted(os.listdir(root)):
        got = hatch_sets(os.path.join(root, entry)) if \
            os.path.isdir(os.path.join(root, entry)) else {}
        if got:
            found[entry] = got
    return found


def read_gene(path):
    """(rho_tor, T[keV], n[1e19 m^-3]) by column order, as everything else uses."""
    d = np.loadtxt(path, comments="#")
    return d[:, 0], d[:, 2], d[:, 3]


def ramp_weight(rhot, top, mid):
    return (np.tanh((rhot - top) / (mid - top)) + 1) / 2


def fit_transform(rhot, base, scaled):
    """(alpha, topped, midped, rms, flat_ratio, flat_spread) from one pair."""
    m = (rhot > 0.02) & (rhot < 0.98) & (base > 0) & (scaled > 0)
    flat = scaled[m] / base[m]
    denom = np.log(base[m] / base[-1])
    ok = np.abs(denom) > 1e-6
    expo = np.log(scaled[m][ok] / scaled[-1]) / denom[ok]
    x = rhot[m][ok]

    def resid(p):
        return 1 + (p[0] - 1) * ramp_weight(x, p[1], p[2]) - expo

    best = None
    for a0 in (0.7, 1.3, 2.0):
        for t0 in (0.0, 0.3, 0.8):
            r = least_squares(resid, [a0, t0, 1.0],
                              bounds=([0.1, -2.0, 0.5], [5.0, 0.95, 3.0]))
            if best is None or r.cost < best.cost:
                best = r
    return (best.x[0], best.x[1], best.x[2],
            float(np.sqrt(np.mean(best.fun ** 2))),
            float(np.median(flat)), float(np.std(flat)))


ROOT = hatch_root()
RUNS = {}
FITS = {}
if ROOT is None:
    print("no hatch tree on this machine; set HATCH_ROOT to run this section.")
    print("tried:", [d for d in HATCH_ROOT_CANDIDATES if d])
else:
    RUNS = hatch_runs(ROOT)
    print(f"hatch tree: {ROOT}")
    for run, sets in RUNS.items():
        print(f"\n=== {run}   sets: "
              f"{', '.join(s.lstrip('_') or 'reference' for s in sorted(sets))} ===")
        print("%-10s %-6s %8s %8s %8s %8s   %10s %8s"
              % ("set", "prof", "alpha", "topped", "midped", "rms",
                 "flat rat", "spread"))
        for suffix, trio in sorted(sets.items()):
            if not suffix:
                continue
            for spec, col in (("e", "Te"), ("e", "ne"), ("i", "Ti"), ("i", "ni")):
                rhot, Tb, nb = read_gene(sets[""][spec])
                _, Ts, ns = read_gene(trio[spec])
                base, scaled = (Tb, Ts) if col.startswith("T") else (nb, ns)
                fit = fit_transform(rhot, base, scaled)
                FITS[(run, suffix, col)] = fit
                print("%-10s %-6s %8.4f %8.4f %8.4f %8.4f   %10.4f %8.4f"
                      % ((suffix.lstrip("_"), col) + fit))


### Reproduce his sets with our transforms

The fitted parameters go straight back into `apply_omt` / `apply_omne` on his own
base profiles, and the result is differenced against his scaled file. Sub-percent
means the two are the same operation; a large error at the pedestal with a small
one in the core means the ramp is right and the window is not.

In [ ]:
def as_ds(sets_or_trio):
    """A Dataset on rhot from one {spec: path} trio."""
    rhot, _, _ = read_gene(sets_or_trio["e"])
    ds = xr.Dataset(coords={"rhot": rhot})
    for spec in ("e", "i", "z"):
        _, T, n = read_gene(sets_or_trio[spec])
        ds[{"e": "Te", "i": "Ti", "z": "Tz"}[spec]] = ("rhot", T)
        ds[{"e": "ne", "i": "ni", "z": "nz"}[spec]] = ("rhot", n)
    return ds


def mimic(run, sets, suffix):
    """(our reproduction, (alpha, top, mid)) for one hatch set, or None.

    The knob is taken per family, which is what scaling them individually means:
    a `*T` set is reproduced with apply_omt alone, a `*n` set with apply_omne
    alone, at the parameters fitted from that set's own files.
    """
    is_T = suffix.rstrip().endswith("T")
    key = (run, suffix, "Te" if is_T else "ne")
    if key not in FITS:
        return None
    alpha, top, mid = FITS[key][:3]
    base = as_ds(sets[""])
    if is_T:
        out, _ = apply_omt(base, alpha=alpha, rhot_midped=mid, rhot_topped=top,
                           apply_to_tz=TZ_EQ_TI)
    else:
        out, _ = apply_omne(base, alpha=alpha, rhot_midped=mid, rhot_topped=top)
    return out, (alpha, top, mid)


for run, sets in RUNS.items():
    for suffix, trio in sorted(sets.items()):
        if not suffix:
            continue
        got = mimic(run, sets, suffix)
        if got is None:
            continue
        ours, (alpha, top, mid) = got
        rhot = ours.coords["rhot"].values
        his, base = as_ds(trio), as_ds(sets[""])
        print(f"\n--- {run} / {suffix.lstrip('_')}   reproduced with "
              f"alpha={alpha:.4f}, topped={top:.4f}, midped={mid:.4f} ---")
        print("%-6s %10s %10s" % ("", "max %err", "rms %err"))
        for col in ("Te", "ne", "Ti", "ni"):
            m = (rhot > 0.02) & (rhot < 0.98) & (his[col].values != 0)
            err = 100 * (ours[col].values[m] / his[col].values[m] - 1)
            print("%-6s %10.3f %10.3f"
                  % (col, np.max(np.abs(err)), np.sqrt(np.mean(err ** 2))))

        fig, axes = plt.subplots(2, 4, figsize=(16, 7), squeeze=False)
        for row, (lo, hi) in enumerate((RHO_FULL, RHO_PED)):
            for ax, col in zip(axes[row], ("Te", "ne", "Ti", "ni")):
                m = (rhot >= lo) & (rhot <= hi)
                ax.plot(rhot[m], base[col].values[m], "k:", lw=1.2, label="base")
                ax.plot(rhot[m], his[col].values[m], lw=2.6, alpha=0.35,
                        color="tab:blue", label="hatch")
                ax.plot(rhot[m], ours[col].values[m], "r--", lw=1.2, label="ours")
                ax.set_title(f"{col}   [{lo}-{hi}]", fontsize=9)
                ax.set_xlabel("rho_tor")
                ax.grid(alpha=0.3)
            axes[row][0].legend(fontsize=7)
        fig.suptitle(f"{run} / {suffix.lstrip('_')} — hatch vs our reproduction")
        fig.tight_layout()
        plt.show()


---
## 6. Hand off to cheaseBS

`run_cheasebs_selfscaled_scan.py` does the solving; this notebook only decides
the knobs. That script keeps its own `RAMP_WINDOWS` table, so a window settled on
here has to be copied into it, or passed on the command line as below.

`RUN_CHEASEBS` stays `False`: four discharges times five points at ~40 s per
CHEASE iteration is a batch job, not a notebook cell.

In [ ]:
SCAN = ("$HOME/NSTXU_lithium_study/reshape_convergence/DIIID_verify/"
        "run_cheasebs_selfscaled_scan.py")
CONT = " \\\n    "

# The NERSC root, not discharge_root(): these lines are meant to be pasted into a
# shell there, so a laptop path -- and the backslash os.path.join would put in it
# -- would be wrong twice over.
NERSC_ROOT = DISCHARGE_ROOT_CANDIDATES[0]

for s in SHOTS:
    top, mid = RAMP_WINDOWS.get(s, DEFAULT_WINDOW)
    tag = str(top).replace(".", "p")
    print(f"# {s}")
    print(f"python {SCAN}{CONT}"
          f"--case-dir {NERSC_ROOT}/{s}{CONT}"
          f"--method omn_omt --rhot-topped {top} --rhot-midped {mid}{CONT}"
          f"--outroot $SCRATCH/{s}/omtomn_top{tag} --in-place --max-iter 25\n")

if RUN_CHEASEBS:
    raise NotImplementedError(
        "Run the commands above as a batch job instead: four discharges x "
        f"{len(ALPHAS)} points is hours of CHEASE, not an interactive cell.")